# 实践实验：手写数字识别的神经网络（多分类）

在本练习中，你将使用神经网络识别手写数字 0-9。


# 大纲
- [ 1 - Packages ](#1)
- [ 2 - ReLU Activation](#2)
- [ 3 - Softmax Function](#3)
  - [ Exercise 1](#ex01)
- [ 4 - Neural Networks](#4)
  - [ 4.1 Problem Statement](#4.1)
  - [ 4.2 Dataset](#4.2)
  - [ 4.3 Model representation](#4.3)
  - [ 4.4 Tensorflow Model Implementation](#4.4)
  - [ 4.5 Softmax placement](#4.5)
    - [ Exercise 2](#ex02)

<a name="1"></a>
## 1 - 软件包 

首先，运行下面的单元格，导入本作业中所需的所有软件包。
- [numpy](https://numpy.org/) 是 Python 科学计算的基础软件包。
- [matplotlib](http://matplotlib.org) 是一个流行的 Python 绘图库。
- [tensorflow](https://www.tensorflow.org/) 是一个流行的机器学习平台。

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.activations import linear, relu, sigmoid
%matplotlib widget
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')

import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

from public_tests import * 

from autils import *
from lab_utils_softmax import plt_softmax
np.set_printoptions(precision=2)

<a name="2"></a>
## 2 - ReLU 激活函数
本周介绍了一种新的激活函数：修正线性单元（ReLU）。 
$$ a = max(0,z) \quad\quad\text {# ReLU function} $$

In [2]:
plt_act_trio()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

<img align="right" src="./images/C2_W2_ReLu.png"     style=" width:380px; padding: 10px 20px; " >
右侧课程中的例子展示了 ReLU 的一种应用。在这个例子中，派生出的“认知度”特征不是二值的，而是具有连续的取值范围。sigmoid 最适合开/关或二值的情形，而 ReLU 提供了连续的线性关系。此外，它还有一个“关闭”区间，在该区间内输出为零。     
“关闭”特性使 ReLU 成为一种非线性激活函数。为什么需要这一点？这使得多个单元能够在互不干扰的情况下共同作用于最终的函数。配套的选做实验对此有更深入的探讨。 

<a name="3"></a>
## 3 - Softmax 函数
多分类神经网络会生成 N 个输出，其中选择一个输出作为预测答案。在输出层中，线性函数生成向量 $\mathbf{z}$，并送入 Softmax 函数。Softmax 函数将 $\mathbf{z}$ 转换为概率分布，如下所述。应用 Softmax 后，每个输出都将介于 0 和 1 之间，且所有输出之和为 1。它们可以被解释为概率。Softmax 的输入越大，对应的输出概率就越大。
<center>  <img  src="./images/C2_W2_NNSoftmax.PNG" width="600" />  

Softmax 函数可以写作：
$$a_j = \frac{e^{z_j}}{ \sum_{k=0}^{N-1}{e^{z_k} }} \tag{1}$$

其中 $z = \mathbf{w} \cdot \mathbf{x} + b$，N 是输出层中特征/类别的数量。  

<a name="ex01"></a>
### 练习 1
让我们创建一个 NumPy 实现：

In [7]:
# UNQ_C1
# GRADED CELL: my_softmax

def my_softmax(z):  
    """ Softmax converts a vector of values to a probability distribution.
    Args:
      z (ndarray (N,))  : input data, N features
    Returns:
      a (ndarray (N,))  : softmax of z
    """    
    ### START CODE HERE ### 
    ez = np.exp(z)
    a = ez/np.sum(ez)
    ### END CODE HERE ### 
    return a

In [8]:
z = np.array([1., 2., 3., 4.])
a = my_softmax(z)
atf = tf.nn.softmax(z)
print(f"my_softmax(z):         {a}")
print(f"tensorflow softmax(z): {atf}")

# BEGIN UNIT TEST  
test_my_softmax(my_softmax)
# END UNIT TEST  

my_softmax(z):         [0.03 0.09 0.24 0.64]
tensorflow softmax(z): [0.03 0.09 0.24 0.64]
 All tests passed.


<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    一种实现方法是先用一个 for 循环构建分母，再用第二个循环计算每个输出。
    
```python
def my_softmax(z):  
    N = len(z)
    a =                     # initialize a to zeros 
    ez_sum =                # initialize sum to zero
    for k in range(N):      # loop over number of outputs             
        ez_sum +=           # sum exp(z[k]) to build the shared denominator      
    for j in range(N):      # loop over number of outputs again                
        a[j] =              # divide each the exp of each output by the denominator   
    return(a)
```
<details>
  <summary><font size="3" color="darkgreen"><b>点击查看代码</b></font></summary>
   
```python
def my_softmax(z):  
    N = len(z)
    a = np.zeros(N)
    ez_sum = 0
    for k in range(N):                
        ez_sum += np.exp(z[k])       
    for j in range(N):                
        a[j] = np.exp(z[j])/ez_sum   
    return(a)

Or, a vector implementation:

def my_softmax(z):  
    ez = np.exp(z)              
    a = ez/np.sum(ez)           
    return(a)

```

下面，改变 `z` 输入的值。请特别注意分子中的指数如何放大了数值间的微小差异。另请注意，输出值之和为 1。

In [9]:
plt.close("all")
plt_softmax(my_softmax)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

<a name="4"></a>
## 4 - 神经网络

在上周的作业中，你实现了一个用于二分类的神经网络。本周你将把它扩展到多分类任务，这将使用 softmax 激活函数。


<a name="4.1"></a>
### 4.1 问题描述

在本练习中，你将使用神经网络识别十个手写数字 0-9。这是一个多分类任务，需要从 n 个选项中选择一个。如今，手写数字的自动识别应用非常广泛——从识别邮件信封上的邮政编码，到识别银行支票上书写的金额。 


<a name="4.2"></a>
### 4.2 数据集

你将首先加载该任务的数据集。
- 下面所示的 `load_data()` 函数将数据加载到变量 `X` 和 `y` 中


- 该数据集包含 5000 个手写数字 $^1$ 的训练样本。  

    - 每个训练样本都是该数字的一幅 20 像素 x 20 像素的灰度图像。 
        - 每个像素由一个浮点数表示，表示该位置的灰度强度。 
        - 20 x 20 的像素网格被“展开”成一个 400 维的向量。 
        - 每个训练样本成为数据矩阵 `X` 中的一行。 
        - 这样就得到了一个 5000 x 400 的矩阵 `X`，其中每一行都是一个手写数字图像的训练样本。

$$X = 
\left(\begin{array}{cc} 
--- (x^{(1)}) --- \\
--- (x^{(2)}) --- \\
\vdots \\ 
--- (x^{(m)}) --- 
\end{array}\right)$$ 

- 训练集的第二部分是一个 5000 x 1 维的向量 `y`，其中包含训练集的标签
    - 如果图像是数字 `0`，则标签为 `y = 0`；如果图像是数字 `4`，则标签为 `y = 4`，依此类推。

$^1$<sub> 这是 MNIST 手写数字数据集 (http://yann.lecun.com/exdb/mnist/) 的一个子集</sub>

In [10]:
# load dataset
X, y = load_data()

#### 4.2.1 查看变量
让我们进一步熟悉你的数据集。  
- 一个好的起点是把每个变量打印出来，看看它们包含什么内容。

下面的代码打印变量 `X` 和 `y` 的第一个元素。  

In [ ]:
print ('The first element of X is: ', X[0])

In [11]:
print ('The first element of y is: ', y[0,0])
print ('The last element of y is: ', y[-1,0])

The first element of y is:  0
The last element of y is:  9


#### 4.2.2 检查变量的维度

熟悉数据的另一种方法是查看它的维度。请打印 `X` 和 `y` 的形状，看看你的数据集中有多少个训练样本。

In [12]:
print ('The shape of X is: ' + str(X.shape))
print ('The shape of y is: ' + str(y.shape))

The shape of X is: (5000, 400)
The shape of y is: (5000, 1)


#### 4.2.3 数据可视化

你将从可视化训练集的一个子集开始。 
- 在下面的单元格中，代码从 `X` 中随机选取 64 行，把每一行映射回 20 像素 x 20 像素的灰度图像，并将这些图像一起显示。 
- 每幅图像的标签显示在图像上方 

In [13]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# You do not need to modify anything in this cell

m, n = X.shape

fig, axes = plt.subplots(8,8, figsize=(5,5))
fig.tight_layout(pad=0.13,rect=[0, 0.03, 1, 0.91]) #[left, bottom, right, top]

#fig.tight_layout(pad=0.5)
widgvis(fig)
for i,ax in enumerate(axes.flat):
    # Select random indices
    random_index = np.random.randint(m)
    
    # Select rows corresponding to the random indices and
    # reshape the image
    X_random_reshaped = X[random_index].reshape((20,20)).T
    
    # Display the image
    ax.imshow(X_random_reshaped, cmap='gray')
    
    # Display the label above the image
    ax.set_title(y[random_index,0])
    ax.set_axis_off()
    fig.suptitle("Label, image", fontsize=14)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

<a name="4.3"></a>
### 4.3 模型表示

本作业中将要使用的神经网络如下图所示。 
- 它包含两个使用 ReLU 激活的全连接层，后面跟着一个使用线性激活的输出层。 
    - 回忆一下，我们的输入是数字图像的像素值。
    - 由于图像尺寸为 $20\times20$，因此有 $400$ 个输入  
    
<img src="images/C2_W2_Assigment_NN.png" width="600" height="450">

- 这些参数的维度是按照如下神经网络设定的：第 1 层有 $25$ 个神经元，第 2 层有 $15$ 个神经元，第 3 层有 $10$ 个输出单元，每个数字对应一个输出单元。

    - 回忆一下，这些参数的维度按如下方式确定：
        - 如果网络的某一层有 $s_{in}$ 个神经元，下一层有 $s_{out}$ 个神经元，那么 
            - $W$ 的维度为 $s_{in} \times s_{out}$。
            - $b$ 是一个包含 $s_{out}$ 个元素的向量
  
    - 因此，`W` 和 `b` 的形状为 
        - 第 1 层：`W1` 的形状是 (400, 25)，`b1` 的形状是 (25,)
        - 第 2 层：`W2` 的形状是 (25, 15)，`b2` 的形状是： (15,)
        - 第 3 层：`W3` 的形状是 (15, 10)，`b3` 的形状是： (10,)
>**注意：** 偏置向量 `b` 可以表示为一维 (n,) 或二维 (n,1) 数组。Tensorflow 采用一维表示，本实验也将沿用这一约定： 
                

<a name="4.4"></a>
### 4.4 使用 Tensorflow 实现模型

Tensorflow 模型是逐层构建的。层的输入维度（上文中的 $s_{in}$）会自动为你计算。你只需指定层的*输出维度*，这就决定了下一层的输入维度。第一层的输入维度由下面 `model.fit` 语句中指定的输入数据大小推导得出。 
>**注意：** 也可以添加一个输入层来指定第一层的输入维度。例如：  
`tf.keras.Input(shape=(400,)),    #specify input shape`  
我们在这里会包含这一设置，以便说明模型的一些尺寸信息。

<a name="4.5"></a>
### 4.5 Softmax 的放置位置
如课程和选做的 softmax 实验中所讲，在训练期间，如果把 softmax 与损失函数放在一起，而不是放在输出层中，数值稳定性会更好。这对*构建*模型和*使用*模型都有影响。  
构建时：  
* 最后一个 Dense 层应使用“linear”激活，这实际上就是没有激活。 
* `model.compile` 语句会通过包含 `from_logits=True` 来表明这一点。
`loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) `  
* 这不会影响目标值的形式。对于 SparseCategorialCrossentropy，目标值就是期望的数字 0-9。

使用模型时：
* 输出不是概率。如果需要输出概率，可以再应用一个 softmax 函数。

<a name="ex02"></a>
### 练习 2

下面，使用 Keras 的 [Sequential model](https://keras.io/guides/sequential_model/) 和 [Dense Layer](https://keras.io/api/layers/core_layers/dense/)，并采用 ReLU 激活函数，构建上述三层网络。

In [16]:
# UNQ_C2
# GRADED CELL: Sequential model
tf.random.set_seed(1234) # for consistent results
model = Sequential(
    [               
        ### START CODE HERE ### 
        tf.keras.layers.InputLayer((400,)),
        tf.keras.layers.Dense(25, activation="relu", name="L1"),
        tf.keras.layers.Dense(15, activation="relu", name="L2"),
        tf.keras.layers.Dense(10, activation="linear", name="L3")
        ### END CODE HERE ### 
    ], name = "my_model" 
)
model.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))

In [17]:
model.summary()

Model: "my_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 L1 (Dense)                  (None, 25)                10025     
                                                                 
 L2 (Dense)                  (None, 15)                390       
                                                                 
 L3 (Dense)                  (None, 10)                160       
                                                                 
Total params: 10,575
Trainable params: 10,575
Non-trainable params: 0
_________________________________________________________________


<details>
  <summary><font size="3" color="darkgreen"><b>期望输出（点击展开）</b></font></summary>
`model.summary()` 函数会显示模型的有用摘要。注意，除非指定了名称，否则层的名称可能有所不同，因为它们是自动生成的。    
    
```
Model: "my_model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
=================================================================
L1 (Dense)                   (None, 25)                10025     
_________________________________________________________________
L2 (Dense)                   (None, 15)                390       
_________________________________________________________________
L3 (Dense)                   (None, 10)                160       
=================================================================
Total params: 10,575
Trainable params: 10,575
Non-trainable params: 0
_________________________________________________________________
```

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
```python
tf.random.set_seed(1234)
model = Sequential(
    [               
        ### START CODE HERE ### 
        tf.keras.Input(shape=(400,)),     # @REPLACE 
        Dense(25, activation='relu', name = "L1"), # @REPLACE 
        Dense(15, activation='relu',  name = "L2"), # @REPLACE  
        Dense(10, activation='linear', name = "L3"),  # @REPLACE 
        ### END CODE HERE ### 
    ], name = "my_model" 
)
``` 

In [18]:
# BEGIN UNIT TEST     
test_model(model, 10, 400)
# END UNIT TEST     

All tests passed!


摘要中显示的参数数量对应于权重数组和偏置数组中元素的数量，如下所示。

让我们进一步检查这些权重，验证 Tensorflow 生成的维度与我们上面计算的一致。

In [19]:
[layer1, layer2, layer3] = model.layers

In [20]:
#### Examine Weights shapes
W1,b1 = layer1.get_weights()
W2,b2 = layer2.get_weights()
W3,b3 = layer3.get_weights()
print(f"W1 shape = {W1.shape}, b1 shape = {b1.shape}")
print(f"W2 shape = {W2.shape}, b2 shape = {b2.shape}")
print(f"W3 shape = {W3.shape}, b3 shape = {b3.shape}")

W1 shape = (400, 25), b1 shape = (25,)
W2 shape = (25, 15), b2 shape = (15,)
W3 shape = (15, 10), b3 shape = (10,)


**期望输出**
```
W1 shape = (400, 25), b1 shape = (25,)  
W2 shape = (25, 15), b2 shape = (15,)  
W3 shape = (15, 10), b3 shape = (10,)
```

下面的代码：
* 定义了一个损失函数 `SparseCategoricalCrossentropy`，并通过添加 `from_logits=True`）来表明 softmax 应包含在损失计算中
* 定义了一个优化器。一个流行的选择是自适应矩估计（Adam），课程中已经介绍过。

In [21]:
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
)

history = model.fit(
    X,y,
    epochs=40
)

Epoch 1/40
157/157 [==============================] - 1s 2ms/step - loss: 1.7094
Epoch 2/40
157/157 [==============================] - 0s 2ms/step - loss: 0.7480
Epoch 3/40
157/157 [==============================] - 0s 2ms/step - loss: 0.4428
Epoch 4/40
157/157 [==============================] - 0s 2ms/step - loss: 0.3463
Epoch 5/40
157/157 [==============================] - 0s 2ms/step - loss: 0.2977
Epoch 6/40
157/157 [==============================] - 0s 2ms/step - loss: 0.2630
Epoch 7/40
157/157 [==============================] - 0s 2ms/step - loss: 0.2361
Epoch 8/40
157/157 [==============================] - 0s 2ms/step - loss: 0.2131
Epoch 9/40
157/157 [==============================] - 0s 2ms/step - loss: 0.2004
Epoch 10/40
157/157 [==============================] - 0s 2ms/step - loss: 0.1805
Epoch 11/40
157/157 [==============================] - 0s 2ms/step - loss: 0.1692
Epoch 12/40
157/157 [==============================] - 0s 2ms/step - loss: 0.1580
Epoch 13/40
157/157 [====

#### 轮次与批次
在上面的 `compile` 语句中，`epochs` 的数量被设置为 100。这表示整个数据集在训练过程中要被完整使用 100 次。在训练过程中，你会看到描述训练进度的输出，如下所示：
```
Epoch 1/100
157/157 [==============================] - 0s 1ms/step - loss: 2.2770
```
第一行 `Epoch 1/100` 描述模型当前正在运行第几个轮次（epoch）。为了提高效率，训练数据集会被划分成多个“批次”（batch）。Tensorflow 中批次默认的大小是 32。我们的数据集有 5000 个样本，大约对应 157 个批次。第二行 `157/157 [====` 的标记描述的是已执行到哪个批次。

#### 损失（代价）
在课程 1 中，我们学习了通过监测代价来跟踪梯度下降的进展。理想情况下，代价会随着算法迭代次数的增加而下降。Tensorflow 将代价称为 `loss`。在上面 `model.fit` 执行时，你可以看到每个轮次都显示了损失。[.fit](https://www.tensorflow.org/api_docs/python/tf/keras/Model) 方法会返回包括损失在内的各种指标，这些指标被保存在上面的 `history` 变量中。可以用它来绘制损失曲线进行检查，如下所示。

In [22]:
plot_loss_tf(history)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

#### 预测 
要进行预测，可以使用 Keras 的 `predict`。下面，X[1015] 包含一幅数字 2 的图像。

In [23]:
image_of_two = X[1015]
display_digit(image_of_two)

prediction = model.predict(image_of_two.reshape(1,400))  # prediction

print(f" predicting a Two: \n{prediction}")
print(f" Largest Prediction index: {np.argmax(prediction)}")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

 predicting a Two: 
[[ -7.99  -2.23   0.77  -2.41 -11.66 -11.15  -9.53  -3.36  -4.42  -7.17]]
 Largest Prediction index: 2


最大的输出是 prediction[2]，表明预测的数字是“2”。如果问题只需要做出选择，这就足够了。可以使用 NumPy 的 [argmax](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html) 来选取它。如果问题需要概率，则需要进行 softmax 运算：

In [24]:
prediction_p = tf.nn.softmax(prediction)

print(f" predicting a Two. Probability vector: \n{prediction_p}")
print(f"Total of predictions: {np.sum(prediction_p):0.3f}")

 predicting a Two. Probability vector: 
[[1.42e-04 4.49e-02 8.98e-01 3.76e-02 3.61e-06 5.97e-06 3.03e-05 1.44e-02
  5.03e-03 3.22e-04]]
Total of predictions: 1.000


要返回一个表示预测目标的整数，你需要的是最大概率对应的索引。这可以通过 NumPy 的 [argmax](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html) 函数实现。

In [25]:
yhat = np.argmax(prediction_p)

print(f"np.argmax(prediction_p): {yhat}")

np.argmax(prediction_p): 2


让我们对随机抽取的 64 个数字比较预测值与标签。这需要一点运行时间。

In [26]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# You do not need to modify anything in this cell

m, n = X.shape

fig, axes = plt.subplots(8,8, figsize=(5,5))
fig.tight_layout(pad=0.13,rect=[0, 0.03, 1, 0.91]) #[left, bottom, right, top]
widgvis(fig)
for i,ax in enumerate(axes.flat):
    # Select random indices
    random_index = np.random.randint(m)
    
    # Select rows corresponding to the random indices and
    # reshape the image
    X_random_reshaped = X[random_index].reshape((20,20)).T
    
    # Display the image
    ax.imshow(X_random_reshaped, cmap='gray')
    
    # Predict using the Neural Network
    prediction = model.predict(X[random_index].reshape(1,400))
    prediction_p = tf.nn.softmax(prediction)
    yhat = np.argmax(prediction_p)
    
    # Display the label above the image
    ax.set_title(f"{y[random_index,0]},{yhat}",fontsize=10)
    ax.set_axis_off()
fig.suptitle("Label, yhat", fontsize=14)
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

让我们看看其中一些错误。 
>注意：增加训练轮数可以消除该数据集上的错误。

In [27]:
print( f"{display_errors(model,X,y)} errors out of {len(X)} images")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

15 errors out of 5000 images


### 恭喜！
你已经成功构建并使用神经网络完成了多分类任务。